# Classify objects from measured features

**Purpose.** Fit a supervised tabular classifier to per-object measurements and apply it to the full dataset.

**Recommended use.** Use when morphology, intensity, or spatial measurements provide an interpretable representation of the phenotype.

**Primary outputs.** Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.ml.generate_ml_scores`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.generate_ml_scores)

```python
generate_ml_scores(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.ml import generate_ml_scores

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.ml.generate_ml_scores`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.generate_ml_scores)


#### Labels & Classes

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`dataset_mode`** *(optional)* — (str) - How training classes are defined: 'metadata' splits crops by well metadata, 'annotation' by the values in one or more annotation columns of png_list. Either way the classes themselves are set in the Classes editor, which names a column and a value per class. A settings file written before the 'measurement' basis was removed still loads: it is read as 'annotation', which is what its threshold rules resolved to after writing their label column. Any other value aborts and returns no dataset. Default 'metadata'.
- **`location_column`** *(optional)* — (str) - Metadata column searched for the positive_control and negative_control values when labelling rows for ML training, normally 'columnID' or 'rowID'. Set 'rowID' when your controls run along plate rows instead of columns. It is overwritten with annotation_column whenever that is set. Default 'columnID'.
- **`positive_control`** *(conditionally required)* — (str) - Identifier of the positive-control class. In ML screening it is the value in location_column (e.g. 'c2') whose objects are labelled class 1 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '239740') matched against coefficient names to tag them 'pc' in the results and volcano plot. Defaults 'c2' and '239740' respectively.
- **`negative_control`** *(conditionally required)* — (str) - Identifier of the negative-control class. In ML screening it is the value in location_column (e.g. 'c1') whose objects are labelled class 0 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '233460') matched against coefficient names to tag them 'nc' in the results and volcano plot. Defaults 'c1' and '233460' respectively.
- **`annotation_column`** *(conditionally required)* — (str) - Integer column of the png_list table holding manual class calls. The Annotate app adds it with ALTER TABLE if missing and writes labels into it. It is the ground truth when dataset_mode is 'annotation', and the fallback when annotation_columns is unset. Setting it while leaving dataset_mode unset also SELECTS annotation mode, which is how an old settings file keeps working. Default None.

#### Feature Preparation

- **`channel_of_interest`** *(optional)* — (int, list, or str) - What the model is allowed to look at. Pick one channel to train on that channel alone, several to train on the combination, or shape to train on the outlines. Pick nothing and the model sees every measurement. Colocalisation belongs to both channels it measures, so one channel brings its relationships with the others along. It also chooses the channel recruitment is measured on. Default 3 in the machine learning steps, 1 or 2 elsewhere.
- **`exclude`** *(optional)* — (str or list) - Names of measurement columns to drop from the feature set before UMAP embedding or ML training, applied after the channel_of_interest selection. Use it to remove features that leak the label or swamp the embedding. It does not filter database rows; use exclude_rows for that. Default None keeps every feature.
- **`nuclei_limit`** *(optional)* — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do NOT pass False: it is read as 0 and removes every cell, leaving an empty analysis rather than an error. Default None.
- **`pathogen_limit`** *(optional)* — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.
- **`remove_highly_correlated_features`** *(optional)* — (bool) - In the machine-learning feature table, drop any feature whose absolute Pearson correlation with an already-kept feature exceeds 0.95, applied after the channel_of_interest filter. Leave it on so redundant measurements do not split importance scores and slow fitting; turn it off only when you need every original column. Default True. Note the UMAP path uses remove_highly_correlated instead.
- **`remove_low_variance_features`** *(optional)* — (bool) - Drop numeric features whose variance across objects falls below 0.01 before model fitting -- near-constant columns that carry no discriminative signal but still cost time and dilute importance rankings. Turn it off only when your features live on a very small numeric scale, where genuine signal can fall under that fixed cut-off. Default True.
- **`min_cell_count`** *(optional)* — (int) - Wells with fewer than this many cells are dropped. In a regression it is scored objects and the well is left out of the fit; in the machine-learning screen it is measured cells and the well is left out of the plate heatmap, whose pivot is then filled with 0, so an excluded well renders at the bottom of the colour scale rather than blank. Raising it removes noisy, sparsely imaged wells at the cost of power. Set 0 to switch it off. Default 100 for a regression, 25 for the screen.

#### Plate & Batch Correction

- **`batch_correction`** *(optional)* — (str) - Plate/batch correction applied before Image UMAP, ML screen classification or phenotype regression. 'none' leaves measurements alone; 'center' removes each plate's mean shift; 'zscore' aligns plate means and variances; 'robust_zscore' uses median/MAD and tolerates outliers; 'combat' models the batch effect while protecting the terms named in batch_covariate_column. Correct when plates were stained or imaged separately; leave off when they were not, since every method removes real signal that happens to align with plate. See spacr.batch_correction.correct_batch_effects. Default 'none'.
- **`batch_column`** *(optional)* — (str) - Metadata column that identifies independent acquisition batches, normally 'plateID'. Every analyzed row must have a value and at least batch_min_samples rows must occur in each batch. Use an acquisition date or instrument ID only if that is the nuisance source you intend to remove. Default 'plateID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_column`** *(optional)* — (str or None) - Metadata column containing reference-control labels for control_center, normally 'columnID' for plate controls. It is ignored by center, zscore, robust_zscore, and none. Blank follows col_to_compare in Image UMAP or location_column in Classify (ML); regression defaults to 'columnID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_values`** *(optional)* — (str, number, list or None) - Reference/negative-control value(s) in batch_control_column used by control_center. Each plate needs at least batch_min_samples matching rows. Image UMAP falls back to neg and Classify (ML) to negative_control when this field is blank; regression requires an explicit value. Default varies by module. API: spacr.batch_correction.correct_batch_effects.
- **`batch_covariate_column`** *(optional)* — (str, list or None) - Metadata column(s) naming the biology combat must PROTECT, e.g. 'condition' or 'condition,timepoint'. combat estimates the batch effect from the residuals after these terms, so anything NOT listed is treated as noise and removed along with the plate effect. Leave your treatment out of this list and combat will quietly delete the effect you are measuring. See spacr.batch_correction.correct_batch_effects. Default None.
- **`batch_combat_mean_only`** *(optional)* — (bool) - True corrects only the additive batch shift and leaves each batch's scale alone. Use it when the plates differ in level but not in spread, or when a batch has too few rows for a stable variance estimate. False (the default) corrects both location and scale, which is standard ComBat. Ignored by every method other than combat. API: spacr.batch_correction.correct_batch_effects.
- **`batch_min_samples`** *(optional)* — (int) - Minimum number of rows required in every batch, and minimum matching reference controls per batch for control_center. Correction stops with an actionable error below this threshold because a one- or two-object plate estimate is unstable. Default 3. API: spacr.batch_correction.correct_batch_effects.
- **`batch_missing_control`** *(optional)* — (str) - Policy when control_center cannot find enough reference controls on a plate: 'error' stops rather than silently mixing corrected and raw plates; 'skip' leaves that plate unchanged and records a warning. Default 'error'. API: spacr.batch_correction.correct_batch_effects.

#### Classifier & Validation

- **`model_type_ml`** *(optional)* — (str) - Which classifier ml_analysis fits to separate positive- from negative-control wells and rank per-object features by permutation importance. One of xgboost (default), lightgbm, catboost, random_forest, extra_trees, gradient_boosting, logistic_regression, svm, mlp; lightgbm and catboost need their optional packages. reg_alpha, reg_lambda and learning_rate only affect the boosted models; logistic_regression is a good linear sanity check.
- **`n_estimators`** *(optional)* — (int) - Number of trees or boosting rounds in the tabular ML classifier - n_estimators for RandomForest/ExtraTrees/XGBoost/LightGBM, iterations for CatBoost, max_iter for HistGradientBoosting. More rounds keep improving fit up to a plateau while training time grows linearly; boosted models can overfit past it. Default 1000.
- **`learning_rate`** *(optional)* — (float) - Step size passed to the optimizer. Too high and the loss spikes or flatlines at chance; too low and training crawls or settles in a poor minimum. 1e-3 suits training from scratch, while 1e-4 to 1e-5 is safer when fine-tuning ImageNet weights (init_weights=True). The chosen schedule decays this starting value over the run. Default 0.001.
- **`test_size`** *(optional)* — (float) - Fraction of the labelled single-object rows held out as the test split in the tabular ML classifier; the remainder trains the model. Raise it for a more trustworthy accuracy estimate, lower it when labelled data is scarce and you need the rows for training. Valid 0-1, default 0.2 (20% test).
- **`cross_validation`** *(optional)* — (bool) - Score the classifier with 5-fold stratified cross-validation instead of a single train/test split, so every control object receives an out-of-fold prediction and an optimal probability threshold is picked per fold. Gives a far more stable accuracy estimate on small control sets, at roughly 5x the training time. Default True.
- **`reg_alpha`** *(optional)* — (float) - L1 penalty on leaf weights for the gradient-boosted classifier (XGBoost and LightGBM; ignored by the other model_type_ml choices). Raising it drives more leaf weights to exactly zero, shrinking the model and its effective feature set - raise it when training accuracy far exceeds test accuracy. Any value &gt;= 0. Default 0.1.
- **`reg_lambda`** *(optional)* — (float) - L2 penalty on leaf weights for the gradient-boosted classifier (XGBoost, LightGBM, and CatBoost's l2_leaf_reg). Raising it shrinks all weights smoothly rather than zeroing them, damping the influence of any single feature and curbing overfitting, at the risk of underfitting if pushed too far. Any value &gt;= 0. Default 1.0.

#### Feature Selection & Importance

- **`prune_features`** *(optional)* — (bool) - Before training, keep only the top_features columns with the highest ANOVA F-score against the control labels (sklearn SelectKBest with f_classif). Speeds up fitting and can curb overfitting on small control sets, but discards features the model might have used and scores each feature in isolation, ignoring interactions. Default False.
- **`top_features`** *(optional)* — (int) - Feature cap in the ML screen analysis: how many rows the feature-importance and permutation-importance bar plots show, and how many top-ranked features the SHAP refit and its summary plot use. It is also the k of the SelectKBest pruning applied before the model is fitted, but only when prune_features is True - with prune_features at its default False the classifier trains on every feature and this is reporting/SHAP scope only. Raise for a fuller picture, lower for readable plots. Default 30.
- **`n_repeats`** *(optional)* — (int) - Number of times each feature is randomly shuffled when computing permutation importance for the ML classifier. More repeats shrink the error bars on the importance ranking but cost an extra full prediction pass per feature per repeat. Default 10; drop to 3-5 for a quick look at wide feature tables.

#### Output & Database

- **`save_to_db`** *(optional)* — (bool) - After ML screen analysis, write the per-object model scores back into measurements.db as a 'predictions' column on the png_list table, matched on prcfo. Enable when you want to sort, filter or plot objects by score in the GUI; the CSV result files are written either way. Default False.

#### Plots & Heatmaps

- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and to plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') keep intensity differences honest; 'gray' matches how the raw microscope data looks. Any registered matplotlib name works, with an '_r' suffix to reverse it. Default 'inferno' for image plots, 'viridis' for plate heatmaps.
- **`heatmap_feature`** *(optional)* — (str) - Numeric column that is aggregated per well and color-mapped in the plate heatmap after ML scoring, e.g. 'predictions' for the classifier score or 'recruitment' for the pathogen/cytoplasm intensity ratio. Must be a numeric column of the scored dataframe or the run raises ValueError listing the valid names. Default 'predictions'.
- **`grouping`** *(optional)* — (str) - How per-object values collapse to one number per well in the plate heatmap: 'mean' averages heatmap_feature over the objects in a well, 'sum' totals them, 'count' ignores the feature and colors wells by object count. Use 'count' to spot uneven seeding or dropout, 'mean' for phenotype strength. Default 'mean'; any other value raises ValueError.
- **`min_max`** *(optional)* — (str) - Color limits for the plate heatmap: 'allq' scales to the 2nd-98th percentile of well values so a handful of extreme wells cannot flatten the rest, 'all' scales to the true min and max. A two-element list is also accepted, where floats are read as quantiles and integers as absolute vmin/vmax. Default 'allq'.

#### Runtime & Reliability

- **`verbose`** *(optional)* — (bool) - Print extra run detail instead of the minimal log: the resolved settings table, the channel and model choices per object type, per-table row counts, and how many objects survive each filter. It only adds console output, so turn it on when object counts come out unexpected and you need to see which stage removed them. The default differs per pipeline -- True for mask, UMAP, screen analysis, barcode mapping and Cellpose training; False for measure, the plotting helpers and regression.
- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Labels & Classes
    # Required settings
    'src': 'path',
    # Conditionally required settings
    'positive_control': 'c2',
    'negative_control': 'c1',
    'annotation_column': None,
    # Optional settings
    'dataset_mode': 'metadata',
    'location_column': 'columnID',

    # Feature Preparation
    # Optional settings
    'channel_of_interest': 3,
    'exclude': None,
    'nuclei_limit': True,
    'pathogen_limit': 3,
    'remove_highly_correlated_features': True,
    'remove_low_variance_features': True,
    'min_cell_count': 25,

    # Plate & Batch Correction
    # Optional settings
    'batch_correction': 'none',
    'batch_column': 'plateID',
    'batch_control_column': None,
    'batch_control_values': None,
    'batch_covariate_column': None,
    'batch_combat_mean_only': False,
    'batch_min_samples': 3,
    'batch_missing_control': 'error',

    # Classifier & Validation
    # Optional settings
    'model_type_ml': 'xgboost',
    'n_estimators': 1000,
    'learning_rate': 0.001,
    'test_size': 0.2,
    'cross_validation': True,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,

    # Feature Selection & Importance
    # Optional settings
    'prune_features': False,
    'top_features': 30,
    'n_repeats': 10,

    # Output & Database
    # Optional settings
    'save_to_db': False,

    # Plots & Heatmaps
    # Optional settings
    'cmap': 'viridis',
    'heatmap_feature': 'predictions',
    'grouping': 'mean',
    'min_max': 'allq',

    # Runtime & Reliability
    # Optional settings
    'verbose': True,
    'n_jobs': -1,
}

In [ ]:
generate_ml_scores(settings)

## Outputs and next steps

Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)